Cortex AI Observability lab notebook with setup and cost attribution queries
*Co-authored with CoCo*

Cortex AI Observability lab notebook with setup and cost attribution queries
*Co-authored with CoCo*

# Cortex AI Observability: Surface Identification & Cost Attribution

**Duration:** 30-45 minutes  
**Scenario:** You're an AI platform admin who needs to understand adoption patterns across Cortex AI surfaces and attribute costs to the right teams.

**What you'll do:**
1. Explore Snowflake's AI usage views (Agent, CoWork, Cortex Code)
2. Identify which surfaces are generating traffic (external API, SQL function, Admin UI, CoWork)
3. Parse model-level token costs and cache hit ratios
4. Attribute costs by user, agent, and cost-center tags
5. Build a unified cross-surface dashboard view
6. Analyze query-level compute attribution

**Prerequisites:** Run `setup.sql` before starting this notebook. It creates the database, warehouse, unified view, and simulated fallback data.

---
## Section 1: Setup & Connection

Establish a Snowpark session and verify access to ACCOUNT_USAGE views.

In [ ]:
# Connection setup — works in both Snowsight notebooks and local Jupyter
import os
import pandas as pd

try:
    # Snowsight notebook: session already exists
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    # Local Jupyter: create session from environment or connection config
    from snowflake.snowpark import Session
    connection_params = {
        "account": os.environ.get("SNOWFLAKE_ACCOUNT"),
        "user": os.environ.get("SNOWFLAKE_USER"),
        "password": os.environ.get("SNOWFLAKE_PASSWORD"),
        "role": os.environ.get("SNOWFLAKE_ROLE", "ACCOUNTADMIN"),
        "warehouse": "OBSERVABILITY_LAB_WH",
        "database": "OBSERVABILITY_LAB",
        "schema": "PUBLIC",
    }
    session = Session.builder.configs(connection_params).create()

# Create lab objects if they don't exist
session.sql("CREATE DATABASE IF NOT EXISTS OBSERVABILITY_LAB").collect()
session.sql("CREATE WAREHOUSE IF NOT EXISTS OBSERVABILITY_LAB_WH WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS OBSERVABILITY_LAB.PUBLIC").collect()

# Set context
session.sql("USE DATABASE OBSERVABILITY_LAB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("USE WAREHOUSE OBSERVABILITY_LAB_WH").collect()
print(f"Connected as: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")
print(f"Role: {session.sql('SELECT CURRENT_ROLE()').collect()[0][0]}")

In [ ]:
# Verify access to ACCOUNT_USAGE views
USE_LIVE_DATA = True
try:
    result = session.sql("SELECT COUNT(*) AS cnt FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY").to_pandas()
    row_count = result['CNT'].iloc[0]
    print(f"Access confirmed. CORTEX_AGENT_USAGE_HISTORY has {row_count:,} rows.")
    USE_LIVE_DATA = True
except Exception as e:
    print(f"Cannot access ACCOUNT_USAGE: {e}")
    print("Falling back to SIMULATED_AI_USAGE table for this lab.")
    USE_LIVE_DATA = False

---
## Section 2: Explore the Usage Views

Snowflake exposes Cortex AI usage across three ACCOUNT_USAGE views:

| View | What it captures |
|------|------------------|
| `CORTEX_AGENT_USAGE_HISTORY` | All Cortex Agent API calls (external, SQL function, Admin UI) |
| `SNOWFLAKE_INTELLIGENCE_USAGE_HISTORY` | CoWork (Snowflake Intelligence) interactions |
| `CORTEX_CODE_CLI_USAGE_HISTORY` | Cortex Code CLI sessions |

Key columns to understand:
- **METADATA** — VARIANT column containing `interaction_interface`, model info, tool calls
- **TOKENS** — Total tokens consumed in the request
- **TOKEN_CREDITS** — Credit cost attributed to token consumption
- **TOKENS_GRANULAR** — Array with per-model token breakdowns (input, output, cache)

In [ ]:
# Explore Cortex Agent Usage History
if USE_LIVE_DATA:
    df_agent = session.sql("""
        SELECT 
            START_TIME,
            USER_NAME,
            AGENT_NAME,
            METADATA:"interaction_interface"::STRING AS INTERFACE,
            TOKENS,
            TOKEN_CREDITS,
            TOKENS_GRANULAR
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
        ORDER BY START_TIME DESC
        LIMIT 5
    """).to_pandas()
else:
    df_agent = session.sql("""
        SELECT 
            REQUEST_TIMESTAMP,
            USER_NAME,
            AGENT_NAME,
            INTERFACE,
            TOKENS,
            TOKEN_CREDITS
        FROM SIMULATED_AI_USAGE
        WHERE SURFACE != 'COWORK'
        ORDER BY REQUEST_TIMESTAMP DESC
        LIMIT 5
    """).to_pandas()

print("=== Cortex Agent Usage (sample) ===")
df_agent

In [ ]:
# Explore Snowflake Intelligence (CoWork) Usage History
if USE_LIVE_DATA:
    df_cowork = session.sql("""
        SELECT 
            START_TIME,
            USER_NAME,
            SNOWFLAKE_INTELLIGENCE_NAME,
            TOKENS,
            TOKEN_CREDITS
        FROM SNOWFLAKE.ACCOUNT_USAGE.SNOWFLAKE_INTELLIGENCE_USAGE_HISTORY
        ORDER BY START_TIME DESC
        LIMIT 5
    """).to_pandas()
else:
    df_cowork = session.sql("""
        SELECT 
            REQUEST_TIMESTAMP,
            USER_NAME,
            AGENT_NAME AS SNOWFLAKE_INTELLIGENCE_NAME,
            TOKENS,
            TOKEN_CREDITS
        FROM SIMULATED_AI_USAGE
        WHERE SURFACE = 'COWORK'
        ORDER BY REQUEST_TIMESTAMP DESC
        LIMIT 5
    """).to_pandas()

print("=== Snowflake Intelligence / CoWork Usage (sample) ===")
df_cowork

---
## Section 3: Surface Identification

The `interaction_interface` field in METADATA tells you *how* the agent was invoked:

| Interface Value | Meaning |
|----------------|----------|
| `external` | MCP clients, programmatic REST API, third-party integrations |
| `sql_function` | Called via SQL (e.g., `SELECT SNOWFLAKE.CORTEX.AGENT(...)`) |
| `agent_admin_ui` | Snowsight Agent Admin playground |
| `rest_api` | Direct REST API calls |

Let's group by interface to see which surfaces drive the most usage.

In [ ]:
# Surface identification: group by interaction_interface
if USE_LIVE_DATA:
    df_surface = session.sql("""
        SELECT 
            METADATA:"interaction_interface"::STRING AS INTERFACE,
            COUNT(*) AS REQUEST_COUNT,
            SUM(TOKEN_CREDITS) AS TOTAL_CREDITS,
            SUM(TOKENS) AS TOTAL_TOKENS
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
        WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        GROUP BY 1
        ORDER BY REQUEST_COUNT DESC
    """).to_pandas()
else:
    df_surface = session.sql("""
        SELECT 
            INTERFACE,
            COUNT(*) AS REQUEST_COUNT,
            SUM(TOKEN_CREDITS) AS TOTAL_CREDITS,
            SUM(TOKENS) AS TOTAL_TOKENS
        FROM SIMULATED_AI_USAGE
        GROUP BY 1
        ORDER BY REQUEST_COUNT DESC
    """).to_pandas()

print("=== Request Volume by Interface (Last 30 Days) ===")
print(df_surface.to_string(index=False))

In [ ]:
# Visualize: bar chart of request volume by interface
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(df_surface['INTERFACE'], df_surface['REQUEST_COUNT'], color='steelblue')
axes[0].set_xlabel('Request Count')
axes[0].set_title('Requests by Interface')

axes[1].barh(df_surface['INTERFACE'], df_surface['TOTAL_CREDITS'], color='darkorange')
axes[1].set_xlabel('Total Credits')
axes[1].set_title('Credit Consumption by Interface')

plt.tight_layout()
plt.show()

print("\nKey insight: 'external' typically dominates — these are MCP clients,")
print("programmatic integrations, and third-party apps calling your agents.")

---
## Section 4: Distinguishing CoWork vs Agent API

The same agent can be invoked from multiple surfaces. CoWork (Snowflake Intelligence) traffic
appears in a *separate* view — `SNOWFLAKE_INTELLIGENCE_USAGE_HISTORY` — even though it may
call the same underlying agents.

The `SNOWFLAKE_INTELLIGENCE_NAME` column identifies which CoWork instance was used.

In [ ]:
# Compare Agent API traffic vs CoWork traffic
if USE_LIVE_DATA:
    df_compare = session.sql("""
        SELECT 'Agent API' AS SOURCE, COUNT(*) AS REQUESTS, SUM(TOKEN_CREDITS) AS CREDITS
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
        WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        UNION ALL
        SELECT 'CoWork' AS SOURCE, COUNT(*) AS REQUESTS, SUM(TOKEN_CREDITS) AS CREDITS
        FROM SNOWFLAKE.ACCOUNT_USAGE.SNOWFLAKE_INTELLIGENCE_USAGE_HISTORY
        WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
    """).to_pandas()
else:
    df_compare = session.sql("""
        SELECT 'Agent API' AS SOURCE, COUNT(*) AS REQUESTS, SUM(TOKEN_CREDITS) AS CREDITS
        FROM SIMULATED_AI_USAGE
        WHERE SURFACE != 'COWORK'
        UNION ALL
        SELECT 'CoWork' AS SOURCE, COUNT(*) AS REQUESTS, SUM(TOKEN_CREDITS) AS CREDITS
        FROM SIMULATED_AI_USAGE
        WHERE SURFACE = 'COWORK'
    """).to_pandas()

print("=== Agent API vs CoWork Traffic ===")
print(df_compare.to_string(index=False))
print("\nNote: CoWork traffic goes through Snowflake Intelligence and is logged")
print("separately. The same agent may appear in both views with different entry points.")

---
## Section 5: Cost Attribution by Model

The `TOKENS_GRANULAR` column is an array of objects, each representing a model used
during the request. Each entry contains:
- `model_name` — which LLM was invoked
- `input_tokens`, `output_tokens` — consumed tokens
- `cache_read_input_tokens` — tokens served from prompt cache (cheaper)
- `token_credits` — credits charged for this model call

We use LATERAL FLATTEN to explode the array and aggregate by model.

In [ ]:
# Cost attribution by model using TOKENS_GRANULAR
if USE_LIVE_DATA:
    df_model = session.sql("""
        SELECT 
            g.value:"model_name"::STRING AS MODEL_NAME,
            COUNT(*) AS CALL_COUNT,
            SUM(g.value:"input_tokens"::NUMBER) AS TOTAL_INPUT_TOKENS,
            SUM(g.value:"output_tokens"::NUMBER) AS TOTAL_OUTPUT_TOKENS,
            SUM(g.value:"cache_read_input_tokens"::NUMBER) AS TOTAL_CACHE_TOKENS,
            SUM(g.value:"token_credits"::NUMBER(10,6)) AS TOTAL_CREDITS,
            ROUND(
                SUM(g.value:"cache_read_input_tokens"::NUMBER) / 
                NULLIF(SUM(g.value:"input_tokens"::NUMBER), 0) * 100, 1
            ) AS CACHE_HIT_PCT
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY,
            LATERAL FLATTEN(input => TOKENS_GRANULAR) g
        WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        GROUP BY 1
        ORDER BY TOTAL_CREDITS DESC
    """).to_pandas()
    print("=== Credit Consumption by Model (Last 30 Days) ===")
    print(df_model.to_string(index=False))
    print(f"\nCache hit ratio insight: higher cache_hit_pct means more prompt-cache")
    print(f"reuse, which reduces effective cost per request.")
else:
    print("TOKENS_GRANULAR is not available in simulated data.")
    print("In live data, you'd use LATERAL FLATTEN on the TOKENS_GRANULAR array")
    print("to see per-model breakdowns including cache hit ratios.")
    print("\nExample query:")
    print("""
    SELECT 
        g.value:"model_name"::STRING AS MODEL_NAME,
        SUM(g.value:"input_tokens"::NUMBER) AS TOTAL_INPUT_TOKENS,
        SUM(g.value:"cache_read_input_tokens"::NUMBER) AS CACHE_TOKENS,
        SUM(g.value:"token_credits"::NUMBER(10,6)) AS CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY,
        LATERAL FLATTEN(input => TOKENS_GRANULAR) g
    GROUP BY 1
    ORDER BY CREDITS DESC
    """)

---
## Section 6: Cost Attribution by User & Agent

Identify the top consumers by user and agent combination, then visualize
daily credit consumption trends.

In [ ]:
# Top 10 consumers by user + agent
if USE_LIVE_DATA:
    df_top = session.sql("""
        SELECT 
            USER_NAME,
            AGENT_NAME,
            COUNT(*) AS REQUEST_COUNT,
            SUM(TOKEN_CREDITS) AS TOTAL_CREDITS
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
        WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        GROUP BY 1, 2
        ORDER BY TOTAL_CREDITS DESC
        LIMIT 10
    """).to_pandas()
else:
    df_top = session.sql("""
        SELECT 
            USER_NAME,
            AGENT_NAME,
            COUNT(*) AS REQUEST_COUNT,
            SUM(TOKEN_CREDITS) AS TOTAL_CREDITS
        FROM SIMULATED_AI_USAGE
        WHERE AGENT_NAME IS NOT NULL
        GROUP BY 1, 2
        ORDER BY TOTAL_CREDITS DESC
        LIMIT 10
    """).to_pandas()

print("=== Top 10 Consumers (User + Agent) ===")
print(df_top.to_string(index=False))

In [ ]:
# Daily credit consumption trend
if USE_LIVE_DATA:
    df_daily = session.sql("""
        SELECT 
            DATE_TRUNC('day', START_TIME) AS DAY,
            SUM(TOKEN_CREDITS) AS DAILY_CREDITS
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
        WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        GROUP BY 1
        ORDER BY 1
    """).to_pandas()
else:
    df_daily = session.sql("""
        SELECT 
            DATE_TRUNC('day', REQUEST_TIMESTAMP) AS DAY,
            SUM(TOKEN_CREDITS) AS DAILY_CREDITS
        FROM SIMULATED_AI_USAGE
        GROUP BY 1
        ORDER BY 1
    """).to_pandas()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_daily['DAY'], df_daily['DAILY_CREDITS'], marker='o', linewidth=2)
ax.set_xlabel('Date')
ax.set_ylabel('Credits')
ax.set_title('Daily Cortex AI Credit Consumption (30 Days)')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\nTotal credits over period: {df_daily['DAILY_CREDITS'].sum():.4f}")
print(f"Average daily credits: {df_daily['DAILY_CREDITS'].mean():.4f}")

---
## Section 7: Tag-Based Cost Center Grouping

If your account uses **user tags** or **agent tags** (VARIANT arrays), you can
attribute costs to organizational cost centers using LATERAL FLATTEN.

This section demonstrates the pattern using simulated data. In production,
tags come from `USER_TAGS` in the usage history views.

In [ ]:
# Tag-based cost center grouping using LATERAL FLATTEN
if USE_LIVE_DATA:
    df_costcenter = session.sql("""
        SELECT 
            t.value:"tag_value"::STRING AS COST_CENTER,
            COUNT(*) AS REQUEST_COUNT,
            SUM(s.TOKEN_CREDITS) AS TOTAL_CREDITS,
            SUM(s.TOKENS) AS TOTAL_TOKENS
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY s,
            LATERAL FLATTEN(input => s.USER_TAGS) t
        WHERE t.value:"tag_name"::STRING = 'cost-center'
          AND s.START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        GROUP BY 1
        ORDER BY TOTAL_CREDITS DESC
    """).to_pandas()
else:
    df_costcenter = session.sql("""
        SELECT 
            t.value:"tag_value"::STRING AS COST_CENTER,
            COUNT(*) AS REQUEST_COUNT,
            SUM(s.TOKEN_CREDITS) AS TOTAL_CREDITS,
            SUM(s.TOKENS) AS TOTAL_TOKENS
        FROM SIMULATED_AI_USAGE s,
            LATERAL FLATTEN(input => s.USER_TAGS) t
        WHERE t.value:"tag_name"::STRING = 'cost-center'
        GROUP BY 1
        ORDER BY TOTAL_CREDITS DESC
    """).to_pandas()

print("=== Credit Attribution by Cost Center ===")
if df_costcenter.empty:
    print("No cost-center tags found in USER_TAGS. Ensure users have tags set.")
    print("Tags can be set via: ALTER USER <user> SET TAG cost_center = 'value';")
else:
    print(df_costcenter.to_string(index=False))
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(df_costcenter['TOTAL_CREDITS'], labels=df_costcenter['COST_CENTER'],
           autopct='%1.1f%%', startangle=90)
    ax.set_title('Credit Distribution by Cost Center')
    plt.show()

---
## Section 8: Build Unified Dashboard View

The `V_UNIFIED_AI_USAGE` view (created in setup.sql) normalizes all three usage
surfaces into a single queryable interface. Let's use it to see the full picture.

In [ ]:
# Query the unified view (or build one inline from account usage)
if USE_LIVE_DATA:
    df_unified = session.sql("""
        SELECT 
            SURFACE,
            COUNT(*) AS REQUEST_COUNT,
            SUM(TOKEN_CREDITS) AS TOTAL_CREDITS,
            COUNT(DISTINCT USER_NAME) AS UNIQUE_USERS
        FROM (
            SELECT 'CORTEX_AGENT' AS SURFACE, USER_NAME, TOKEN_CREDITS
            FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
            WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
            UNION ALL
            SELECT 'SNOWFLAKE_INTELLIGENCE' AS SURFACE, USER_NAME, TOKEN_CREDITS
            FROM SNOWFLAKE.ACCOUNT_USAGE.SNOWFLAKE_INTELLIGENCE_USAGE_HISTORY
            WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        )
        GROUP BY 1
        ORDER BY TOTAL_CREDITS DESC
    """).to_pandas()
else:
    df_unified = session.sql("""
        SELECT 
            SURFACE,
            COUNT(*) AS REQUEST_COUNT,
            SUM(TOKEN_CREDITS) AS TOTAL_CREDITS,
            COUNT(DISTINCT USER_NAME) AS UNIQUE_USERS
        FROM SIMULATED_AI_USAGE
        GROUP BY 1
        ORDER BY TOTAL_CREDITS DESC
    """).to_pandas()

# Calculate percentage distribution
total = df_unified['TOTAL_CREDITS'].sum()
df_unified['PCT_OF_TOTAL'] = (df_unified['TOTAL_CREDITS'] / total * 100).round(1)

print("=== Unified AI Usage by Surface ===")
print(df_unified.to_string(index=False))
print(f"\nTotal credits across all surfaces: {total:.4f}")

In [ ]:
# Visualize unified surface distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: requests by surface
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
axes[0].bar(df_unified['SURFACE'], df_unified['REQUEST_COUNT'], color=colors[:len(df_unified)])
axes[0].set_xlabel('Surface')
axes[0].set_ylabel('Request Count')
axes[0].set_title('Request Volume by AI Surface')
axes[0].tick_params(axis='x', rotation=30)

# Pie chart: credit distribution
axes[1].pie(df_unified['TOTAL_CREDITS'], labels=df_unified['SURFACE'],
           autopct='%1.1f%%', colors=colors[:len(df_unified)], startangle=90)
axes[1].set_title('Credit Distribution by Surface')

plt.tight_layout()
plt.show()

---
## Section 9: Query-Level Attribution

`QUERY_ATTRIBUTION_HISTORY` provides warehouse compute costs attributed to individual
queries. By filtering on `QUERY_TAG`, you can isolate costs from Cortex/agent workloads
and attribute them to specific applications or teams.

In [ ]:
# Query-level compute attribution for AI workloads
if USE_LIVE_DATA:
    df_query_attr = session.sql("""
        SELECT 
            q.QUERY_TAG,
            COUNT(*) AS QUERY_COUNT,
            SUM(a.CREDITS_ATTRIBUTED_COMPUTE) AS COMPUTE_CREDITS
        FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY a
        JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY q
            ON a.QUERY_ID = q.QUERY_ID
        WHERE q.QUERY_TAG ILIKE '%cortex%' OR q.QUERY_TAG ILIKE '%agent%'
        AND q.START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        GROUP BY 1
        ORDER BY COMPUTE_CREDITS DESC
        LIMIT 10
    """).to_pandas()
    print("=== Warehouse Compute Attribution for AI Workloads ===")
    print(df_query_attr.to_string(index=False))
else:
    print("QUERY_ATTRIBUTION_HISTORY is not available in simulated data.")
    print("\nIn production, use this query to attribute warehouse compute costs:")
    print("""
    SELECT 
        q.QUERY_TAG,
        COUNT(*) AS QUERY_COUNT,
        SUM(a.CREDITS_ATTRIBUTED_COMPUTE) AS COMPUTE_CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY a
    JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY q
        ON a.QUERY_ID = q.QUERY_ID
    WHERE q.QUERY_TAG ILIKE '%cortex%' OR q.QUERY_TAG ILIKE '%agent%'
    GROUP BY 1
    ORDER BY COMPUTE_CREDITS DESC
    """)
    print("\nTip: Set QUERY_TAG on your application sessions to enable this attribution.")
    print("Example: ALTER SESSION SET QUERY_TAG = 'app=my_agent,team=platform';")

---
## Summary: What You Accomplished

In this lab you completed a full Cortex AI observability workflow:

| Step | What you did |
|------|-------------|
| **Explore** | Queried all three AI usage views and understood their schemas |
| **Surface ID** | Grouped by `interaction_interface` to identify traffic origins |
| **CoWork vs API** | Distinguished CoWork traffic from direct Agent API calls |
| **Model Costs** | Used LATERAL FLATTEN on TOKENS_GRANULAR for per-model attribution |
| **User/Agent** | Identified top consumers and visualized daily trends |
| **Cost Centers** | Attributed credits to teams via tag-based grouping |
| **Unified View** | Queried V_UNIFIED_AI_USAGE for a single-pane-of-glass view |
| **Compute** | Explored query-level warehouse compute attribution |

### What's Next?

- **Set up resource budgets** — Use `CREATE BUDGET` to cap Cortex AI spending per team
- **Build Snowsight dashboards** — Pin these queries as tiles in a monitoring dashboard
- **Configure alerts** — Use `CREATE ALERT` to notify when daily credits exceed thresholds
- **Automate reporting** — Schedule a Task to materialize daily usage summaries
- **Tag governance** — Ensure all agents and users have cost-center tags for clean attribution

In [ ]:
# Optional: Clean up (uncomment to run)
# session.sql("DROP DATABASE IF EXISTS OBSERVABILITY_LAB CASCADE").collect()
# print("Lab resources cleaned up.")